# 05 · DistilBERT Fine-tuning
Loads BERT text splits → tokenises → fine-tunes with a proper **train / val / test** loop including early stopping → evaluates on the held-out test set.

**Prerequisite:** `00_data_preparation.ipynb`

In [ ]:
import os, pickle, random, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (DistilBertTokenizerFast,
                          DistilBertForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from tqdm import tqdm
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_DIR = '../outputs/featured/'
OUTPUT_DIR = '../outputs/distilbert/'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Device: {DEVICE}')

def load(name):
    with open(os.path.join(INPUT_DIR, f'{name}.pkl'), 'rb') as f:
        return pickle.load(f)

X_bert_train = load('X_bert_train')
X_bert_val   = load('X_bert_val')
X_bert_test  = load('X_bert_test')
y_train      = load('y_train').to_numpy()
y_val        = load('y_val').to_numpy()
y_test       = load('y_test').to_numpy()

print(f'Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')


## 1 · Tokenise
`MAX_LEN_BERT=512` — full context window. Reduce to 256 if you hit GPU OOM.

In [ ]:
BERT_MODEL   = 'distilbert-base-uncased'
MAX_LEN_BERT = 512
BERT_BATCH   = 8     # reduce to 4 if OOM
BERT_EPOCHS  = 5     # early stopping will typically fire earlier
BERT_LR      = 2e-5
PATIENCE     = 2

tokenizer = DistilBertTokenizerFast.from_pretrained(BERT_MODEL)

class BertDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(
            texts.tolist(),
            truncation=True, padding=True,
            max_length=MAX_LEN_BERT,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):  return len(self.labels)
    def __getitem__(self, i):
        return {
            'input_ids':      self.enc['input_ids'][i],
            'attention_mask': self.enc['attention_mask'][i],
            'labels':         self.labels[i]
        }

print('Tokenising train …')
train_ds = BertDataset(X_bert_train, y_train)
print('Tokenising val …')
val_ds   = BertDataset(X_bert_val,   y_val)
print('Tokenising test …')
test_ds  = BertDataset(X_bert_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BERT_BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BERT_BATCH, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BERT_BATCH, shuffle=False)
print('Done.')


## 2 · Load Model & Optimiser

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    BERT_MODEL, num_labels=2
).to(DEVICE)

optimizer   = torch.optim.AdamW(model.parameters(), lr=BERT_LR, eps=1e-8)
total_steps = len(train_loader) * BERT_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)
print(f'Total optimiser steps: {total_steps:,}')


## 3 · Training Loop with Early Stopping on Validation F1

In [ ]:
def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, desc='  Train' if train else '  Eval '):
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)
            labels    = batch['labels'].to(DEVICE)
            if train:
                optimizer.zero_grad()
            out  = model(input_ids, attention_mask=attn_mask, labels=labels)
            loss = out.loss
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            total_loss += loss.item() * labels.size(0)
            preds       = out.logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / total
    acc      = correct / total
    f1       = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, acc, f1, np.array(all_preds)

history        = []
best_val_f1    = -1.0
epochs_no_impr = 0
best_state     = None

for epoch in range(1, BERT_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{BERT_EPOCHS}')
    tr_loss,  tr_acc,  tr_f1,  _    = run_epoch(model, train_loader, train=True)
    val_loss, val_acc, val_f1, _    = run_epoch(model, val_loader,   train=False)

    history.append(dict(epoch=epoch,
                        tr_loss=tr_loss, tr_acc=tr_acc, tr_f1=tr_f1,
                        val_loss=val_loss, val_acc=val_acc, val_f1=val_f1))

    print(f'  Train  loss={tr_loss:.4f}  acc={tr_acc:.4f}  f1={tr_f1:.4f}')
    print(f'  Val    loss={val_loss:.4f}  acc={val_acc:.4f}  f1={val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1    = val_f1
        epochs_no_impr = 0
        best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'  ✓ New best val F1: {best_val_f1:.4f}')
    else:
        epochs_no_impr += 1
        print(f'  No improvement ({epochs_no_impr}/{PATIENCE})')
        if epochs_no_impr >= PATIENCE:
            print('Early stopping triggered.')
            break

model.load_state_dict(best_state)
print(f'\nTraining complete. Best Val F1: {best_val_f1:.4f}')


## 4 · Evaluate on Test Set

In [ ]:
_, _, _, y_pred_bert = run_epoch(model, test_loader, train=False)

metrics = dict(
    Accuracy  = accuracy_score(y_test, y_pred_bert),
    Precision = precision_score(y_test, y_pred_bert, average='weighted'),
    Recall    = recall_score(y_test, y_pred_bert, average='weighted'),
    F1        = f1_score(y_test, y_pred_bert, average='weighted'),
)
print('\n=== DistilBERT (Test) ===')
for k, v in metrics.items():
    print(f'  {k:10s}: {v:.4f}')
print()
print(classification_report(y_test, y_pred_bert, target_names=['Fake', 'Real']))

cm = confusion_matrix(y_test, y_pred_bert)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fake','Real'], yticklabels=['Fake','Real'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — DistilBERT')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cm_DistilBERT.png'), dpi=120)
plt.show()


## 5 · Training Curve

In [ ]:
hist_df = pd.DataFrame(history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_df['epoch'], hist_df['tr_loss'], label='Train')
ax1.plot(hist_df['epoch'], hist_df['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(hist_df['epoch'], hist_df['tr_f1'], label='Train')
ax2.plot(hist_df['epoch'], hist_df['val_f1'], label='Val')
ax2.set_title('F1'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.suptitle('DistilBERT Training Curve', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'curve_distilbert.png'), dpi=120)
plt.show()


## 6 · Save Artefacts

In [ ]:
model.save_pretrained(os.path.join(OUTPUT_DIR, 'model_distilbert'))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, 'model_distilbert'))

with open(os.path.join(OUTPUT_DIR, 'pred_distilbert.pkl'), 'wb') as f:
    pickle.dump(y_pred_bert, f)

with open(os.path.join(OUTPUT_DIR, 'metrics_distilbert.json'), 'w') as f:
    json.dump({'test': metrics, 'history': history}, f, indent=2)

print('Saved: model_distilbert/, pred_distilbert.pkl, metrics_distilbert.json')
